# 04b Team 3, part 2: data preparation for the correction ladder (E11)

**Purpose.** Prepare compact inputs for re-implementing Team 3's hybrid NCF + LSTM model (`04c_team3_ladder.ipynb`) and correcting its flaws one at a time. This notebook is CPU work (parsing the raw files, about 20 minutes); the training notebook only needs the artifacts it writes, so paid GPU time on Colab is spent on training alone.

**What Team 3 did** (their repository, `Data Preparation/` and `Models/3.1_Hybrid_NCF_with_LSTM.ipynb`), and what we prepare for each part of the ladder:

| Team 3 step | Their version (rung R0) | Correct version (later rungs) |
|---|---|---|
| Candidate lists | all positives + 4 negatives each (`1. Sampling`) | full lists |
| History | products outside the **sampled candidate lists of all users** deleted, which depends on labels, including validation labels (`filtered`) | the same cut from **training lists only** (`filtered_train`, rung R0b: no validation labels involved); then the full history |
| Time gaps | final gap to the search lost (lists zipped with the one-longer time list); meta features drop one more gap | all gaps including the gap to the search |
| Candidate lookup | candidate index from the **history** vocabulary (candidates never seen in a history get 0), used to index attribute tables built in a **different** order | one consistent candidate vocabulary |
| User meta features | 15 per user: category diversity (L1-L4), history length, average log L1-category size, click / cart / order ratios, KMeans cluster (k = 4, standardised as a number), mean / std / max gap, total / average time decay (`exp(-0.02 days) x type weight`) | the same 15 on the full history, plus days since the last action |

Rung R5 also gets our non-personalised candidate features (G1 relevance, G2 popularity) from feature set v2.

**Data.** Training: the 40,000-search dev subsample of `split_v1` (10% of it held out as an inner validation set for checkpointing, as Team 3 did with their own validation split). Evaluation: our 20,000 validation searches. Candidates are shuffled within each search with the same per-search seed as feature set v2.

**Approximations** (their notebooks depend on Drive artifacts we do not have): the L1-category sizes come from the full catalogue, not their filtered one; vocabularies are built from training searches only (items unseen in training map to the padding index, where Team 3's untrained random vectors carried no information either).

**Output:** `data/processed/team3/` (locally) or `MyDrive/BT4222 Group Project/processed/team3/` (Colab): `train_dev.npz`, `validation.npz`, `lookups.npz`, `meta.json`. Copy the folder to the Drive location before running `04c` on Colab.

In [1]:
from collections import Counter
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
    OUT = Path("/content/drive/MyDrive/BT4222 Group Project/processed/team3")
else:
    DATA_DIR = REPO_ROOT / "data"
    OUT = REPO_ROOT / "data/processed/team3"
PROCESSED = REPO_ROOT / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)
BEHAVIOUR, CATALOGUE = DATA_DIR / "user_behavior_data.txt", DATA_DIR / "product_meta_data.txt"

sys.path.insert(0, str(REPO_ROOT))
from scripts.jd_data import iter_records, SECONDS_PER_DAY

SEED, MAX_LEN, NUM_TIME_BINS, LAMBDA = 4222, 200, 10, 0.02
TYPE_INDEX = {"CLICK": 1, "CART": 2, "ORD": 3, "FLW": 4}
TYPE_WEIGHT = {"CLICK": 1.0, "FLW": 1.5, "CART": 2.0, "ORD": 3.0}  # Team 3's ordinal encoding
R5_FEATURES = ["query_len", "n_candidates", "title_shared", "title_coverage", "title_jaccard", "bm25", "title_len",
               "query_in_brand", "query_in_cate3", "in_catalogue", "pop_exposures", "pop_engaged_rate", "pop_history"]

assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
SPLITS = {"train_dev": assignment.index[assignment.dev_subsample].to_numpy(),
          "validation": assignment.index[assignment.split == "validation"].to_numpy()}
inner_val = set(np.random.default_rng([SEED, 11]).choice(SPLITS["train_dev"], len(SPLITS["train_dev"]) // 10,
                                                          replace=False).tolist())
print({k: len(v) for k, v in SPLITS.items()}, "| inner validation", len(inner_val))

{'train_dev': 40000, 'validation': 20000} | inner validation 4000


## 1. Parse the searches and apply Team 3's sampling

In [2]:
target = {int(s): name for name, ids in SPLITS.items() for s in ids}
records = []
for session_id, fields in iter_records(BEHAVIOUR):
    if session_id not in target:
        continue
    candidates = np.array(fields[1].split("_"))
    labels = np.array([int(float(v)) for v in fields[2].split("_")], dtype=np.int8)
    order = np.random.default_rng([SEED, session_id]).permutation(len(candidates))  # same as feature set v2
    candidates, labels = candidates[order], labels[order]
    pos, neg = np.flatnonzero(labels > 0), np.flatnonzero(labels == 0)
    k = min(4 * len(pos), len(neg))
    sample_rng = np.random.default_rng([SEED, 3, session_id])
    sampled = np.zeros(len(labels), bool)
    sampled[pos] = True
    if k:
        sampled[sample_rng.choice(neg, size=k, replace=False)] = True
    records.append(dict(session_id=session_id, split=target[session_id], candidates=candidates, labels=labels,
                        sampled=sampled, past=fields[4].split("_"), types=fields[5].split("_"),
                        gaps=np.array(fields[6].split("_"), dtype=np.int64)))
kept_wids = {w for r in records for w, s in zip(r["candidates"], r["sampled"]) if s}  # Team 3's catalogue cut (all users)
kept_train = {w for r in records if r["split"] == "train_dev" for w, s in zip(r["candidates"], r["sampled"]) if s}
print(f"searches {len(records):,}; products kept by Team 3's cut: {len(kept_wids):,} (all users), "
      f"{len(kept_train):,} (training lists only)")

searches 60,000; products kept by Team 3's cut: 489,410 (all users), 334,471 (training lists only)


## 2. History variants

- **filtered** (R0): only actions on products in `kept_wids` (sampled lists of all users, as Team 3); each kept action keeps its own gap to the previous action; the gap to the search is lost.
- **filtered_train** (R0b-R2): the same, but with the cut taken from training lists only.
- **full** (R3+): all actions, gaps to the previous action, and the gap to the search for the meta features.

Both keep the most recent 200 actions in the sequence, as Team 3 did.

In [3]:
for r in records:
    gaps_prev = r["gaps"][:-1]  # gap of each action to the previous one (first is 0)
    past, types = np.array(r["past"]), np.array(r["types"])
    keep = np.array([w in kept_wids for w in r["past"]], dtype=bool)
    keep_train = np.array([w in kept_train for w in r["past"]], dtype=bool)
    r["hist"] = {
        "full": (past, types, gaps_prev),
        "filtered": (past[keep], types[keep], gaps_prev[keep]),
        "filtered_train": (past[keep_train], types[keep_train], gaps_prev[keep_train]),
    }
    r["gap_to_search"] = int(r["gaps"][-1])
lengths = pd.DataFrame({v: [len(r["hist"][v][0]) for r in records] for v in ("full", "filtered", "filtered_train")})
print("history length, mean:", lengths.mean().round(1).to_dict(), "| searches with empty history:",
      (lengths == 0).sum().to_dict())

history length, mean: {'full': 152.8, 'filtered': 43.3, 'filtered_train': 35.5} | searches with empty history: {'full': 0, 'filtered': 2709, 'filtered_train': 3490}


## 3. Catalogue: attributes and L1-category sizes

In [4]:
start = time.time()
needed = {w for r in records for w in r["candidates"]} | {w for r in records for w in r["past"]}
attributes, cat1_size = {}, Counter()
with CATALOGUE.open(encoding="utf-8") as handle:
    next(handle)
    for line in handle:
        fields = line.rstrip("\r\n").split("\t")
        if fields[4] not in ("", "-1"):
            cat1_size[fields[4]] += 1
        if fields[0] in needed:
            attributes[fields[0]] = tuple(fields[j] if fields[j] not in ("", "-1") else None for j in (2, 12, 4, 6, 8, 10))
log_cat1 = {c: np.log10(n) for c, n in cat1_size.items()}
print(f"attributes for {len(attributes) / len(needed):.1%} of needed products; {time.time() - start:.0f} s")

attributes for 94.7% of needed products; 32 s


## 4. Vocabularies (from training searches only; 0 = padding / unseen)

- `cand_vocab`: candidate products of the training lists (correct lookup).
- `hist_vocab[variant]`: history products of the training searches for each history variant.
- **Bug variant (R0, R0b):** Team 3 indexed candidates with the history vocabulary (unseen candidates -> 0) and used that index on attribute tables built in candidate-vocabulary order. We reproduce exactly that: `cand_bug_idx = hist_vocab["filtered"][wid]` (R0) and `cand_bug_idx_train = hist_vocab["filtered_train"][wid]` (R0b), looked up in the candidate-ordered attribute tables. Because Team 3's cut keeps all positives but only four negatives each, "has a history-vocabulary index" is itself correlated with the label; with the cut over all users this includes validation labels.
- Attribute vocabularies (brand, shop, L1-L4) from training candidates; time bins: 10 log-spaced bins over training gaps, as Team 3 built them.

In [5]:
train = [r for r in records if r["split"] == "train_dev"]
cand_vocab = {w: i + 1 for i, w in enumerate(sorted({w for r in train for w in r["candidates"]}))}
hist_vocab = {v: {w: i + 1 for i, w in enumerate(sorted({w for r in train for w in r["hist"][v][0]}))}
              for v in ("full", "filtered", "filtered_train")}
attr_vocab = [{} for _ in range(6)]  # brand, shop, c1, c2, c3, c4
for w in cand_vocab:
    for j, value in enumerate(attributes.get(w, (None,) * 6)):
        if value is not None and value not in attr_vocab[j]:
            attr_vocab[j][value] = len(attr_vocab[j]) + 1
lookup = np.zeros((6, len(cand_vocab) + 1), np.int32)  # [attribute, candidate index]
for w, i in cand_vocab.items():
    for j, value in enumerate(attributes.get(w, (None,) * 6)):
        lookup[j, i] = attr_vocab[j].get(value, 0)
all_train_gaps = np.concatenate([r["hist"]["full"][2] for r in train])
log_gaps = np.log1p(all_train_gaps)
bins = np.round(np.expm1(np.linspace(log_gaps.min(), log_gaps.max(), NUM_TIME_BINS))).astype(np.int64)
print(f"candidate vocab {len(cand_vocab):,}; history vocab sizes { {v: len(h) for v, h in hist_vocab.items()} }; "
      f"attribute vocab sizes {[len(v) for v in attr_vocab]}")

candidate vocab 2,609,460; history vocab sizes {'full': 1888666, 'filtered': 131141, 'filtered_train': 94980}; attribute vocab sizes [120111, 194139, 64, 582, 4949, 6128]


## 5. User meta features (Team 3's 15, computed on each history variant)

In [6]:
def meta_features(wids, types, gaps_prev, gap_to_search, with_search_gap):
    cats = [attributes.get(w, (None,) * 6) for w in wids]
    out = {f"nunique_cats_{d}": len({c[1 + d] for c in cats if c[1 + d] is not None}) for d in range(1, 5)}
    out["history_len"] = len(wids)
    sizes = [log_cat1[c[2]] for c in cats if c[2] is not None and c[2] in log_cat1]
    out["avg_log_cat_1_popularity"] = np.mean(sizes) if sizes else 0.0
    n = max(len(types), 1)
    for name, t in [("click_ratio", "CLICK"), ("cart_ratio", "CART"), ("order_ratio", "ORD")]:
        out[name] = sum(x == t for x in types) / n
    if with_search_gap:  # correct: every gap, ages measured from the search
        g = gaps_prev[1:]
        ages = (np.cumsum(np.r_[gaps_prev[1:], gap_to_search][::-1])[::-1]) / SECONDS_PER_DAY
        weights = np.array([TYPE_WEIGHT[t] for t in types])
    else:  # Team 3: the last real gap is dropped too, ages measured from the last action, last type dropped
        g = gaps_prev[1:-1]
        ages = (np.cumsum(np.r_[gaps_prev[1:], 0][::-1])[::-1])[:-1] / SECONDS_PER_DAY
        weights = np.array([TYPE_WEIGHT[t] for t in types[:-1]])
    out["avg_interaction_time_gap"] = g.mean() if len(g) else 0.0
    out["std_time_gap"] = g.std() if len(g) > 1 else 0.0
    out["max_gap"] = g.max() if len(g) else 0.0
    decay = weights * np.exp(-LAMBDA * ages) if len(weights) else np.zeros(0)
    out["total_time_decay"] = decay.sum()
    out["avg_time_decay"] = decay.mean() if len(decay) else 0.0
    if with_search_gap:
        out["days_since_last_action"] = gap_to_search / SECONDS_PER_DAY
    return out


meta = {}
for variant, with_gap in [("filtered", False), ("filtered_train", False), ("full", True)]:
    frame = pd.DataFrame([meta_features(*r["hist"][variant], r["gap_to_search"], with_gap) for r in records])
    is_train = np.array([r["split"] == "train_dev" for r in records])
    scaler = StandardScaler().fit(frame[is_train])
    base = scaler.transform(frame)
    cluster = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(base[is_train]).predict(base)
    frame.insert(9, "cluster", cluster)  # standardised as a number, as Team 3 did
    meta[variant] = StandardScaler().fit(frame[is_train]).transform(frame).astype(np.float32)
    print(variant, "meta features:", list(frame.columns))

filtered meta features: ['nunique_cats_1', 'nunique_cats_2', 'nunique_cats_3', 'nunique_cats_4', 'history_len', 'avg_log_cat_1_popularity', 'click_ratio', 'cart_ratio', 'order_ratio', 'cluster', 'avg_interaction_time_gap', 'std_time_gap', 'max_gap', 'total_time_decay', 'avg_time_decay']


filtered_train meta features: ['nunique_cats_1', 'nunique_cats_2', 'nunique_cats_3', 'nunique_cats_4', 'history_len', 'avg_log_cat_1_popularity', 'click_ratio', 'cart_ratio', 'order_ratio', 'cluster', 'avg_interaction_time_gap', 'std_time_gap', 'max_gap', 'total_time_decay', 'avg_time_decay']


full meta features: ['nunique_cats_1', 'nunique_cats_2', 'nunique_cats_3', 'nunique_cats_4', 'history_len', 'avg_log_cat_1_popularity', 'click_ratio', 'cart_ratio', 'order_ratio', 'cluster', 'avg_interaction_time_gap', 'std_time_gap', 'max_gap', 'total_time_decay', 'avg_time_decay', 'days_since_last_action']


## 6. Our candidate features for rung R5 (feature set v2: G0-G2)

Counts are log-transformed, missing values set to 0 with the features standardised on training rows.

In [7]:
r5 = {}
for split in SPLITS:
    table = pd.read_parquet(PROCESSED / f"features_v2_{split}.parquet", columns=["session_id", "wid"] + R5_FEATURES)
    r5[split] = table.set_index(["session_id", "wid"])
for c in ["pop_exposures", "pop_history", "title_len", "bm25", "title_shared", "n_candidates"]:
    for split in r5:
        r5[split][c] = np.log1p(r5[split][c])
mean, std = r5["train_dev"].mean(), r5["train_dev"].std().replace(0, 1)
for split in r5:
    r5[split] = ((r5[split] - mean) / std).fillna(0).astype(np.float32)

## 7. Encode and save

In [8]:
def encode_split(split):
    rs = [(i, r) for i, r in enumerate(records) if r["split"] == split]
    out = {"session_id": np.array([r["session_id"] for _, r in rs]),
           "inner_val": np.array([r["session_id"] in inner_val for _, r in rs]),
           **{f"meta_{v}": meta[v][[i for i, _ in rs]] for v in meta}}
    cand_off, cand = [0], {k: [] for k in ("cand_idx", "cand_bug_idx", "cand_bug_idx_train", "label", "sampled")}
    feats = []
    for _, r in rs:
        cand["cand_idx"].append([cand_vocab.get(w, 0) for w in r["candidates"]])
        cand["cand_bug_idx"].append([hist_vocab["filtered"].get(w, 0) for w in r["candidates"]])
        cand["cand_bug_idx_train"].append([hist_vocab["filtered_train"].get(w, 0) for w in r["candidates"]])
        cand["label"].append(r["labels"])
        cand["sampled"].append(r["sampled"])
        feats.append(r5[split].loc[r["session_id"]].reindex(r["candidates"].astype(np.int64)).to_numpy())
        cand_off.append(cand_off[-1] + len(r["candidates"]))
    for k, v in cand.items():
        out[k] = np.concatenate(v).astype(np.int8 if k in ("label", "sampled") else np.int32)
    out["r5_features"] = np.concatenate(feats).astype(np.float32)
    out["cand_offsets"] = np.array(cand_off, np.int64)
    for variant in ("filtered", "filtered_train", "full"):
        off, items, types, tbins = [0], [], [], []
        for _, r in rs:
            wids, ts, gaps = (x[-MAX_LEN:] for x in r["hist"][variant])
            items.append([hist_vocab[variant].get(w, 0) for w in wids])
            types.append([TYPE_INDEX[t] for t in ts])
            tbins.append(np.digitize(gaps, bins))
            off.append(off[-1] + len(wids))
        out[f"hist_{variant}_offsets"] = np.array(off, np.int64)
        out[f"hist_{variant}_item"] = np.concatenate(items).astype(np.int32)
        out[f"hist_{variant}_type"] = np.concatenate(types).astype(np.int8)
        out[f"hist_{variant}_time"] = np.concatenate(tbins).astype(np.int8)
    assert np.isfinite(out["r5_features"]).all()
    return out


start = time.time()
for split in SPLITS:
    data = encode_split(split)
    np.savez_compressed(OUT / f"{split}.npz", **data)
    print(f"{split}: {len(data['session_id']):,} searches, {len(data['label']):,} candidates")
np.savez_compressed(OUT / "lookups.npz", lookup=lookup)
info = {"num_cand": len(cand_vocab) + 1, **{f"num_hist_{v}": len(h) + 1 for v, h in hist_vocab.items()},
        "num_types": len(TYPE_INDEX) + 1,
        "num_time_bins": NUM_TIME_BINS + 1, "num_attr": [len(v) + 1 for v in attr_vocab],
        "meta_dims": {v: int(meta[v].shape[1]) for v in meta}, "r5_features": R5_FEATURES,
        "max_len": MAX_LEN, "seed": SEED}
(OUT / "meta.json").write_text(json.dumps(info, indent=2))
print(json.dumps(info), f"| {time.time() - start:.0f} s")
print("files:", {p.name: f"{p.stat().st_size / 1e6:.0f} MB" for p in OUT.iterdir()})

train_dev: 40,000 searches, 3,573,362 candidates


validation: 20,000 searches, 1,781,573 candidates


{"num_cand": 2609461, "num_hist_full": 1888667, "num_hist_filtered": 131142, "num_hist_filtered_train": 94981, "num_types": 5, "num_time_bins": 11, "num_attr": [120112, 194140, 65, 583, 4950, 6129], "meta_dims": {"filtered": 15, "filtered_train": 15, "full": 16}, "r5_features": ["query_len", "n_candidates", "title_shared", "title_coverage", "title_jaccard", "bm25", "title_len", "query_in_brand", "query_in_cate3", "in_catalogue", "pop_exposures", "pop_engaged_rate", "pop_history"], "max_len": 200, "seed": 4222} | 70 s
files: {'lookups.npz': '29 MB', 'meta.json': '0 MB', 'train_dev.npz': '67 MB', 'validation.npz': '29 MB'}


## Checks

In [9]:
data = np.load(OUT / "validation.npz")
labels, sampled, off = data["label"], data["sampled"].astype(bool), data["cand_offsets"]
assert len(off) == len(data["session_id"]) + 1 and off[-1] == len(labels)
assert (labels[sampled] > 0).sum() == (labels > 0).sum()  # every positive is in the sampled lists
print(f"validation: mean full list {np.diff(off).mean():.1f}, mean sampled list "
      f"{np.add.reduceat(sampled, off[:-1]).mean():.1f}")
print(f"candidates with a correct index (seen in training lists): {(data['cand_idx'] > 0).mean():.1%}; "
      f"with a (buggy) history-vocabulary index: {(data['cand_bug_idx'] > 0).mean():.1%} (cut over all users), "
      f"{(data['cand_bug_idx_train'] > 0).mean():.1%} (cut over training lists)")
engaged = labels > 0
for key in ("cand_bug_idx", "cand_bug_idx_train", "cand_idx"):
    has = data[key] > 0
    print(f"{key}: engaged share among candidates with an index {engaged[has].mean():.3f}, without {engaged[~has].mean():.3f}")

validation: mean full list 89.1, mean sampled list 9.3
candidates with a correct index (seen in training lists): 39.2%; with a (buggy) history-vocabulary index: 13.2% (cut over all users), 9.1% (cut over training lists)


cand_bug_idx: engaged share among candidates with an index 0.089, without 0.011
cand_bug_idx_train: engaged share among candidates with an index 0.060, without 0.017
cand_idx: engaged share among candidates with an index 0.032, without 0.014
